# Credit Card Fraud Detection — Model Development & Evaluation

**Prepared by:** Data Science Team (Fraud Analytics)
**Dataset:** [Credit Card Fraud Detection — mlg-ulb](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) (Kaggle)

This notebook builds an end-to-end pipeline for detecting fraudulent credit card transactions:
exploratory analysis of class imbalance, model training (Logistic Regression + Random Forest),
imbalance-aware evaluation, feature importance, and a discussion of production scalability.

**Contents**
1. Data acquisition & loading
2. Class imbalance analysis
3. Exploratory data analysis (amount & time-of-day)
4. Why accuracy is misleading here
5. Preprocessing, stratified train/test split, and imbalance handling (SMOTE)
6. Model training — Logistic Regression & Random Forest
7. Evaluation — Precision, Recall, F1, AUC-ROC
8. Precision/Recall trade-off discussion
9. Feature importance analysis
10. Scalability discussion (1M transactions/hour)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# imbalanced-learn for SMOTE (pip install imbalanced-learn if not already installed)
from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42


## 1. Data Acquisition

Download the latest version of the dataset directly from Kaggle using `kagglehub`.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)


In [ ]:
import os

# The dataset download contains creditcard.csv
csv_path = os.path.join(path, "creditcard.csv")
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
df.head()


## 2. Class Imbalance Analysis

The `Class` column is the target: `0` = legitimate transaction, `1` = fraudulent transaction.
We quantify how skewed the classes are before doing anything else, since this drives every
downstream modeling decision (metric choice, resampling strategy, evaluation protocol).

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100
legit_pct = (class_counts[0] / len(df)) * 100

print(f"Total transactions: {len(df):,}")
print(f"Legitimate (0): {class_counts[0]:,} ({legit_pct:.3f}%)")
print(f"Fraudulent (1): {class_counts[1]:,} ({fraud_pct:.3f}%)")
print(f"Imbalance ratio (legit:fraud) ≈ {class_counts[0]/class_counts[1]:.0f} : 1")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(x='Class', data=df, ax=ax[0], palette=['#3b82f6', '#ef4444'])
ax[0].set_title("Class Distribution (Counts)")
ax[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
ax[0].set_yscale('log')
ax[0].set_ylabel("Count (log scale)")

ax[1].pie([legit_pct, fraud_pct], labels=['Legitimate', 'Fraud'],
          autopct='%1.3f%%', colors=['#3b82f6', '#ef4444'], startangle=90)
ax[1].set_title("Class Distribution (%)")
plt.tight_layout()
plt.show()


## 3. Exploratory Data Analysis

### 3.1 Transaction Amount Distribution — Fraud vs. Legitimate

In [ ]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(legit['Amount'], bins=50, color='#3b82f6', ax=ax[0], stat='density', kde=True)
ax[0].set_title("Legitimate Transaction Amounts")
ax[0].set_xlim(0, 2000)

sns.histplot(fraud['Amount'], bins=50, color='#ef4444', ax=ax[1], stat='density', kde=True)
ax[1].set_title("Fraudulent Transaction Amounts")
ax[1].set_xlim(0, 2000)

plt.tight_layout()
plt.show()

print("Amount summary — Legitimate:")
print(legit['Amount'].describe())
print("\nAmount summary — Fraud:")
print(fraud['Amount'].describe())


### 3.2 Time-of-Day Analysis

`Time` is seconds elapsed since the first transaction in the dataset (spans ~2 days). We convert it to an hour-of-day to see whether fraud clusters at particular times.

In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(data=df[df['Class'] == 0], x='Hour', bins=24, stat='density',
             color='#3b82f6', label='Legitimate', alpha=0.5, ax=ax)
sns.histplot(data=df[df['Class'] == 1], x='Hour', bins=24, stat='density',
             color='#ef4444', label='Fraud', alpha=0.6, ax=ax)
ax.set_title("Transaction Density by Hour of Day: Fraud vs. Legitimate")
ax.set_xlabel("Hour of Day (0-23)")
ax.legend()
plt.tight_layout()
plt.show()

fraud_rate_by_hour = df.groupby('Hour')['Class'].mean() * 100
print("Fraud rate (%) by hour:")
print(fraud_rate_by_hour.round(3))


## 4. Why Standard Accuracy Is Misleading Here

With fraud representing roughly **0.17%** of transactions, a trivial model that predicts
"legitimate" for *every* transaction would score **~99.83% accuracy** while catching **zero**
fraud — the exact opposite of what a fraud system exists to do.

Accuracy treats both error types as equally costly, but in fraud detection they are not:

- **False Negatives (missed fraud):** direct financial loss, chargebacks, regulatory exposure,
  and erosion of customer trust.
- **False Positives (legitimate transactions flagged as fraud):** customer friction, blocked
  purchases, support costs — real, but generally far less severe than a missed fraud.

Because the positive class is rare, accuracy is dominated by how well the model handles the
*majority* class, which is trivial. It gives no signal about the metric that actually matters:
how many frauds were caught, and at what cost in false alarms. This is why we evaluate with
**Precision, Recall, F1-score, and AUC-ROC** instead, and why accuracy is reported only as
context, never as the headline metric.

## 5. Preprocessing & Stratified Train/Test Split

`Time` and `Amount` are on different scales than the PCA-transformed `V1`–`V28` features, so we
scale them. We then split with **stratification on `Class`** so that the (already rare) fraud
cases are proportionally represented in both the training and test sets — a plain random split
risks leaving too few (or zero) fraud cases in the test set.

In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

feature_cols = [c for c in df.columns if c.startswith('V')] + ['Amount_scaled', 'Time_scaled']
X = df[feature_cols]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train shape: {X_train.shape}, fraud rate: {y_train.mean()*100:.3f}%")
print(f"Test shape:  {X_test.shape}, fraud rate: {y_test.mean()*100:.3f}%")


## 6. Handling Class Imbalance — SMOTE

We apply **SMOTE (Synthetic Minority Over-sampling Technique)** to the *training set only*
(never the test set — that would leak synthetic, near-duplicate fraud patterns into evaluation
and inflate performance). SMOTE generates synthetic fraud examples by interpolating between
existing minority-class neighbors, giving the classifier a more balanced signal to learn from
without simply duplicating rows.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_res).value_counts().to_dict())


## 7. Model Training

We train two models on the SMOTE-resampled training data:

1. **Logistic Regression** — interpretable baseline; coefficients give directional insight into
   fraud drivers.
2. **Random Forest** — a stronger, non-linear ensemble model that typically captures the
   complex fraud patterns in this dataset better, and provides a feature-importance ranking.

(As a robustness check, we also fit Logistic Regression with `class_weight='balanced'` directly
on the *original* imbalanced training data, to compare the reweighting approach against SMOTE.)

In [ ]:
# Logistic Regression trained on SMOTE-balanced data
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_res, y_train_res)

# Logistic Regression with class_weight='balanced' on original (non-resampled) data, for comparison
log_reg_bal = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
log_reg_bal.fit(X_train, y_train)

# Random Forest trained on SMOTE-balanced data
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train_res, y_train_res)

print("Models trained: Logistic Regression (SMOTE), Logistic Regression (class_weight='balanced'), Random Forest (SMOTE)")


## 8. Model Evaluation — Precision, Recall, F1, AUC-ROC

All models are evaluated on the **untouched, original-distribution test set** — this is the
only way to know how they'd perform on real, imbalanced traffic.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"--- {name} ---")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

    return {'name': name, 'y_pred': y_pred, 'y_proba': y_proba,
            'precision': precision, 'recall': recall, 'f1': f1, 'auc': auc}

results = []
results.append(evaluate_model("Logistic Regression (SMOTE)", log_reg, X_test, y_test))
results.append(evaluate_model("Logistic Regression (class_weight='balanced')", log_reg_bal, X_test, y_test))
results.append(evaluate_model("Random Forest (SMOTE)", rf, X_test, y_test))


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(res['name'], fontsize=9)
plt.tight_layout()
plt.show()

# ROC curves
plt.figure(figsize=(7, 6))
for res in results:
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, label=f"{res['name']} (AUC = {res['auc']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


## 9. Which Metric Matters Most? The Recall/Precision Trade-off

**For fraud detection, Recall is generally prioritized over Precision — but neither can be
optimized in isolation, which is why AUC-ROC and F1 matter too.**

- **Recall** (a.k.a. sensitivity, true positive rate) = *of all actual frauds, how many did we
  catch?* A missed fraud (false negative) is a direct dollar loss to the institution, so low
  recall is the costliest failure mode in most fraud programs.
- **Precision** = *of everything we flagged, how much was actually fraud?* Low precision means
  many legitimate transactions get blocked or sent to manual review — annoying and costly in
  operational/support terms, but usually a smaller loss than an uncaught fraud.
- **The trade-off:** raising the model's decision threshold increases precision but lowers
  recall (fewer, more confident flags); lowering the threshold increases recall but drops
  precision (more flags, more false alarms). Because these move in opposite directions, no
  single point captures full performance — this is exactly what the **ROC curve and AUC-ROC**
  summarize: AUC measures the model's ability to rank fraud above legitimate transactions
  across *all* thresholds, independent of any one operating point.
- **F1-score** (harmonic mean of precision and recall) is useful as a single balanced number
  when comparing models, but in production the threshold is usually tuned deliberately —
  e.g., "maximize recall subject to precision ≥ X%" — based on the institution's cost ratio
  between fraud losses and customer-friction costs, rather than optimizing F1 blindly.

In practice, most fraud teams set the decision threshold to **favor recall**, accepting a higher
false-positive rate, and route flagged transactions to a secondary review layer (manual
investigation, step-up authentication, temporary hold) rather than auto-declining — this
captures more fraud while containing the customer-experience cost of false positives.

## 10. Feature Importance / Coefficient Analysis

- For **Logistic Regression**, the coefficient magnitude (on standardized features) indicates
  each feature's directional contribution to the fraud log-odds.
- For **Random Forest**, `feature_importances_` reflects how much each feature reduces impurity
  across the ensemble's trees, aggregated and averaged.

In [ ]:
# Logistic Regression coefficients
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': log_reg.coef_[0]
}).assign(abs_coef=lambda d: d['coefficient'].abs()).sort_values('abs_coef', ascending=False).head(15)

# Random Forest feature importances
rf_imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=coef_df, x='coefficient', y='feature', ax=ax[0],
            palette=['#ef4444' if c < 0 else '#3b82f6' for c in coef_df['coefficient']])
ax[0].set_title("Logistic Regression: Top 15 Coefficients")
ax[0].axvline(0, color='black', linewidth=0.8)

sns.barplot(data=rf_imp_df, x='importance', y='feature', ax=ax[1], color='#10b981')
ax[1].set_title("Random Forest: Top 15 Feature Importances")

plt.tight_layout()
plt.show()

print("Top 10 Random Forest features by importance:")
print(rf_imp_df.head(10).to_string(index=False))


**Interpretation notes:** In this dataset the raw features `V1`–`V28` are PCA components of
the original (undisclosed, sensitive) transaction attributes, so they can't be mapped back to
plain-English variable names — but their *relative* importance still tells us which underlying
signal dimensions matter most. Typically components such as `V14`, `V17`, `V12`, `V10`, and
`V4` rank among the strongest fraud predictors in this dataset, alongside the scaled `Amount`
feature, consistent with published analyses of this data. In a real deployment (without PCA
anonymization), this step would be done on interpretable features — e.g., transaction velocity,
merchant category, geolocation mismatch, device fingerprint — so this analysis would map
directly to actionable fraud-rule inputs.

## 11. Scalability: Handling 1,000,000 Transactions per Hour

1M transactions/hour ≈ **278 transactions/second** sustained, with likely bursty peaks well
above that (e.g., holiday shopping). Getting a scikit-learn notebook model into that pipeline
requires rethinking several layers:

**Inference latency & throughput**
- A single Random Forest or Logistic Regression prediction is sub-millisecond, so the *model*
  itself is rarely the bottleneck — the surrounding pipeline is. Batch scoring (vectorized
  `predict_proba` over micro-batches of a few hundred transactions) is far more efficient than
  scoring one row at a time under a request/response API.
- For strict low-latency, real-time scoring (approve/decline in milliseconds at checkout),
  Logistic Regression's simplicity is an asset; a large Random Forest/XGBoost ensemble may need
  optimization (e.g., ONNX export, `treelite` compilation, or reducing tree count/depth) to hit
  tight SLAs at high QPS.

**Architecture**
- Move from a notebook to a served model behind a **feature store + streaming pipeline**
  (e.g., Kafka/Kinesis ingesting transactions, a feature service computing real-time features
  like rolling velocity counts, and a model server such as **TensorFlow Serving, Triton, or a
  lightweight FastAPI/gRPC service** behind a load balancer with horizontal auto-scaling).
- Stateless model replicas scale horizontally across containers/pods (Kubernetes) to absorb
  278+ TPS and burst traffic; a message queue provides backpressure and buffering during spikes.

**Feature computation at scale**
- Many strong fraud features are *aggregation-based* (spend in last 10 min, distinct merchants
  in last hour, velocity since last transaction). These need a low-latency online feature store
  (e.g., Redis, Feast) rather than recomputing from a data warehouse per request.

**Retraining & drift**
- Fraud patterns shift quickly (adversarial adaptation), so the pipeline needs **scheduled
  retraining** (e.g., daily/weekly) plus **drift monitoring** on both the input feature
  distributions and the model's precision/recall in production (via delayed ground-truth
  labels from confirmed chargebacks/disputes).
- A/B or shadow-mode deployment lets a new model run alongside the current one on live traffic
  before fully cutting over, limiting the blast radius of a regression.

**Cost/latency trade-offs**
- Simpler models (Logistic Regression, shallow trees) are cheaper and faster to serve at this
  volume; ensemble models (Random Forest/XGBoost) generally detect fraud better but cost more
  compute per prediction — a common production pattern is a **cheap first-pass model** to
  triage the bulk of obviously-legitimate traffic, escalating only the harder/ambiguous cases
  to a heavier model or human review, keeping average latency and cost low while preserving
  recall on suspicious transactions.

**Monitoring**
- At this volume, real-time dashboards for prediction latency (p50/p99), throughput, model
  score distribution, and rolling precision/recall estimates are essential to catch both
  infrastructure degradation and model/data drift before they translate into fraud losses.

## 12. Summary

- Fraud is extremely rare (**~0.17%** of transactions), so accuracy is not a meaningful metric;
  **Precision, Recall, F1, and AUC-ROC** were used instead.
- **SMOTE** (and, as a comparison, `class_weight='balanced'`) addressed the class imbalance
  during training, while the **test set retained the true fraud distribution** for an honest
  evaluation.
- Both **Logistic Regression** and **Random Forest** were trained and compared on
  Precision/Recall/F1/AUC-ROC; the ROC curves and confusion matrices above show each model's
  operating trade-offs.
- **Recall is the priority metric** for catching fraud, but it must be balanced against
  Precision to avoid overwhelming customers/analysts with false positives — a threshold and
  review-routing decision, not a pure modeling one.
- **Feature importance/coefficient analysis** highlighted the PCA components with the strongest
  fraud signal.
- Scaling to **1M transactions/hour** is primarily an infrastructure and MLOps problem (serving
  architecture, online feature computation, horizontal scaling, drift monitoring, retraining
  cadence) more than a raw-model-throughput problem, given how fast these models already are
  at inference time.
